# Classification d'avis clients — Dataset **Allociné**

**Dataset :** https://huggingface.co/datasets/tblard/allocine  

**Objectif.** Construire un pipeline de classification d'avis (positif / négatif) à partir de verbatims en français.  
**Cas d’usage** : **suivi de la satisfaction**, **modération** et **priorisation** des retours utilisateurs dans une plateforme média francophone.

**Plan :**
- Chargement des données
- Préparation : filtrage, échantillonnage contrôlé, splits
- 1) Nettoyage & prétraitement
- 2) Représentation (TF‑IDF pour ML, Tokenizer+Embedding pour DL)
- 3) Modèles (Logistic Regression, Linear SVM, CNN 1D)
- 4) Évaluation (accuracy, precision, recall, F1, matrice de confusion)

In [1]:
try:
    import sys, subprocess, pkgutil
    def sh(x):
        import subprocess, sys
        subprocess.run(x, shell=True, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    # Compatibilité fsspec/gcsfs
    sh("pip -q install -U 'fsspec==2025.3.0' 'gcsfs==2025.3.0' datasets")
except Exception as e:
    print("Note:", e)

import sys, platform
import pandas as pd
import tensorflow as tf
print("Python:", sys.version.split()[0])
print("Pandas:", pd.__version__)
print("TensorFlow:", tf.__version__)

# Keras import
from tensorflow import keras
from tensorflow.keras.models import Sequential


Python: 3.12.11
Pandas: 2.2.2
TensorFlow: 2.19.0


In [2]:
from tensorflow import keras

# Installation des dépendances
!pip -q install -U 'fsspec==2025.3.0' 'gcsfs==2025.3.0' datasets

import sys, os, re, random, numpy as np, pandas as pd, matplotlib.pyplot as plt
from datasets import load_dataset

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix, roc_auc_score, roc_curve)

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

print("Python:", sys.version.split()[0])
print("Pandas:", pd.__version__)
print("TensorFlow:", tf.__version__)


Python: 3.12.11
Pandas: 2.2.2
TensorFlow: 2.19.0


In [3]:

# Chargement du dataset
# Le dataset contient des avis en français avec un label binaire: 1=positif, 0=négatif
ds = load_dataset("tblard/allocine")

# On combine titre + revue
def to_df(split):
    d = ds[split]
    df = pd.DataFrame(d)
    if "review" in df.columns and "title" in df.columns:
        df["text"] = (df["title"].fillna("") + " " + df["review"].fillna("")).str.strip()
    elif "review" in df.columns:
        df["text"] = df["review"].fillna("")
    else:
        df["text"] = df["text"].fillna("")
    df = df[["text","label"]].rename(columns={"label":"y"}).copy()
    return df

train_df = to_df("train")
val_df   = to_df("validation")
test_df  = to_df("test")

# Sous‑échantillonner, pour réduire le temps d'entraînement
def balanced_sample(df, n_per_class=20000):
    out = []
    for c in [0,1]:
        part = df[df["y"]==c]
        k = min(n_per_class, len(part))
        out.append(part.sample(n=k, random_state=SEED))
    return pd.concat(out).sample(frac=1.0, random_state=SEED)

train_df = balanced_sample(train_df, n_per_class=20000)
val_df   = balanced_sample(val_df,   n_per_class=2500)
test_df  = balanced_sample(test_df,  n_per_class=5000)

train_df.head(5), train_df["y"].value_counts()


README.md: 0.00B [00:00, ?B/s]

allocine/train-00000-of-00001.parquet:   0%|          | 0.00/60.0M [00:00<?, ?B/s]

allocine/validation-00000-of-00001.parqu(…):   0%|          | 0.00/7.58M [00:00<?, ?B/s]

allocine/test-00000-of-00001.parquet:   0%|          | 0.00/7.58M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/160000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/20000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/20000 [00:00<?, ? examples/s]

(                                                     text  y
 92737   Un chef d'oeuvre sur un spectacle de cirque . ...  1
 39885   Manipulation mentale et lavage de cerveau pour...  0
 103952  La nouvelle vague française avait "à bout de s...  1
 141730  un chef d'œuvre d'ennui pathétique, réalisatio...  0
 128885  Quel gâchis. Ce sont les premiers mots qui me ...  1,
 y
 1    20000
 0    20000
 Name: count, dtype: int64)

## 1) Nettoyage & prétraitement
Règles sobres et efficaces pour du français écrit par des utilisateurs :
- minuscules, normalisation des espaces
- retrait **URLs**, **emails**, **mentions/hashtags**
- retrait **ponctuation** et **chiffres**
- petite liste de **stopwords** FR pour limiter le bruit

In [4]:

STOP_FR = {
    "un","une","des","le","la","les","au","aux","du","de","d","dans","sur","sous","chez","vers",
    "et","ou","mais","donc","or","ni","car",
    "je","tu","il","elle","nous","vous","ils","elles",
    "me","te","se","moi","toi","lui","leur","on","y","en",
    "mon","ton","son","notre","votre","leur","mes","tes","ses","nos","vos","leurs",
    "ce","cet","cette","ces","cela","ça","c","ci","là",
    "etre","suis","es","est","sommes","etes","sont","ai","as","a","avons","avez","ont"
}

import re
def normalize_fr(s: str) -> str:
    s = str(s).lower()
    s = re.sub(r"http\\S+|www\\.\\S+", " URL ", s)
    s = re.sub(r"\\S+@\\S+", " EMAIL ", s)
    s = re.sub(r"[@#]\\w+", " TAG ", s)
    # garder lettres FR de base
    s = re.sub(r"[^a-zàâäéèêëîïôöùûüç\\s]", " ", s)
    s = re.sub(r"\\s+", " ", s).strip()
    toks = [w for w in s.split() if w not in STOP_FR]
    return " ".join(toks)

for df in (train_df, val_df, test_df):
    df["clean"] = df["text"].apply(normalize_fr)

train_df[["text","clean","y"]].head(8)


,text,clean,y
92737,Un chef d'oeuvre sur un spectacle de cirque . ...,chef oeuvre spectacle cirque très beau film av...,1
39885,Manipulation mentale et lavage de cerveau pour...,manipulation mentale lavage cerveau pour appre...,0
103952,"La nouvelle vague française avait ""à bout de s...",nouvelle vague française avait à bout souffle ...,1
141730,"un chef d'œuvre d'ennui pathétique, réalisatio...",chef uvre ennui pathétique réalisation mauvais...,0
128885,Quel gâchis. Ce sont les premiers mots qui me ...,quel gâchis premiers mots qui viennent après v...,1
115948,J'avais lu le roman de Chantal Thomas et je n'...,j avais lu roman chantal thomas n étais pas pa...,1
54457,"après les incroyables deux premiers opus, la s...",après incroyables deux premiers opus surexploi...,0
90239,J'ai tout simplement adoré ce film. Je l'ai vu...,j tout simplement adoré film l vu l avion j es...,1


## 2 + 3) Représentation (TF‑IDF) & Modèles ML
Deux modèles en production : **Logistic Regression** et **Linear SVM**.  
Sélection sur **F1** (validation), apprentissage final sur **train+val**, puis test.

In [5]:

from sklearn.metrics import precision_recall_fscore_support

def bin_metrics(y_true, y_pred, y_score=None):
    acc = accuracy_score(y_true, y_pred)
    pr, rc, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary', zero_division=0)
    out = {"accuracy":acc, "precision":pr, "recall":rc, "f1":f1}
    if y_score is not None:
        try:
            out["roc_auc"] = roc_auc_score(y_true, y_score)
        except Exception:
            pass
    return out

results = {}

tfidf = TfidfVectorizer(max_features=80000, ngram_range=(1,2))

# Logistic Regression
pipe_lr = Pipeline([("tfidf", tfidf), ("clf", LogisticRegression(max_iter=400))])
gs_lr = GridSearchCV(pipe_lr, {"clf__C":[0.5,1.0,2.0]}, cv=3, n_jobs=-1, scoring="f1")
gs_lr.fit(train_df["clean"].values, train_df["y"].values)
pred_val_lr = gs_lr.predict(val_df["clean"].values)
# proba/score
try:
    score_val_lr = gs_lr.predict_proba(val_df["clean"].values)[:,1]
except Exception:
    score_val_lr = None
results["LogReg_val"] = bin_metrics(val_df["y"].values, pred_val_lr, score_val_lr)

# Linear SVM
pipe_svm = Pipeline([("tfidf", tfidf), ("clf", LinearSVC())])
gs_svm = GridSearchCV(pipe_svm, {"clf__C":[0.5,1.0,2.0]}, cv=3, n_jobs=-1, scoring="f1")
gs_svm.fit(train_df["clean"].values, train_df["y"].values)
pred_val_svm = gs_svm.predict(val_df["clean"].values)
# decision_function -> [0,1]
dfc = gs_svm.decision_function(val_df["clean"].values)
dfc_scaled = (dfc - dfc.min())/(dfc.max()-dfc.min()+1e-9)
results["LinearSVM_val"] = bin_metrics(val_df["y"].values, pred_val_svm, dfc_scaled)

results


{'LogReg_val': {'accuracy': 0.9286,
  'precision': 0.9203609258532758,
  'recall': 0.9384,
  'f1': 0.9292929292929293,
  'roc_auc': np.float64(0.9782816)},
 'LinearSVM_val': {'accuracy': 0.9326,
  'precision': 0.9256198347107438,
  'recall': 0.9408,
  'f1': 0.9331481848839516,
  'roc_auc': np.float64(0.9796787199999999)}}

In [6]:

best_name = max(results.items(), key=lambda kv: kv[1]["f1"])[0]
best_estimator = gs_lr if "LogReg" in best_name else gs_svm
print("Meilleur modèle (validation) :", best_name)

# Fit sur train+val et test final
X_all = np.concatenate([train_df["clean"].values, val_df["clean"].values])
y_all = np.concatenate([train_df["y"].values,     val_df["y"].values])
best_estimator.fit(X_all, y_all)

y_pred_test = best_estimator.predict(test_df["clean"].values)
if hasattr(best_estimator, "predict_proba"):
    y_score_test = best_estimator.predict_proba(test_df["clean"].values)[:,1]
elif hasattr(best_estimator, "decision_function"):
    sc = best_estimator.decision_function(test_df["clean"].values)
    y_score_test = (sc - sc.min())/(sc.max()-sc.min()+1e-9)
else:
    y_score_test = None

ml_scores_test = bin_metrics(test_df["y"].values, y_pred_test, y_score_test)
print("Scores test (ML) :", ml_scores_test)

print("\nClassification report (ML):\n",
      classification_report(test_df["y"], y_pred_test, target_names=["négatif","positif"], digits=4))

cm_ml = confusion_matrix(test_df["y"], y_pred_test)
cm_ml


Meilleur modèle (validation) : LinearSVM_val
Scores test (ML) : {'accuracy': 0.9347, 'precision': 0.933572710951526, 'recall': 0.936, 'f1': 0.9347847797862778, 'roc_auc': np.float64(0.9833506799999999)}

Classification report (ML):
               precision    recall  f1-score   support

     négatif     0.9358    0.9334    0.9346      5000
     positif     0.9336    0.9360    0.9348      5000

    accuracy                         0.9347     10000
   macro avg     0.9347    0.9347    0.9347     10000
weighted avg     0.9347    0.9347    0.9347     10000



array([[4667,  333],
       [ 320, 4680]])

## 2) Représentation (DL) & 3) Modèle DL (CNN 1D)
- **Tokenizer** (40k mots) + **padding**
- **Embedding(128)** + **Conv1D(128)** + **GlobalMaxPooling** + **Dropout**
- **Dense sigmoïde**, **EarlyStopping**

In [7]:
from tensorflow.keras.models import Sequential

VOCAB = 40000
tok = Tokenizer(num_words=VOCAB, oov_token="<OOV>")
tok.fit_on_texts(train_df["clean"].values)

def to_seq(x): return tok.texts_to_sequences(x)

seq_tr = to_seq(train_df["clean"].values)
seq_va = to_seq(val_df["clean"].values)
seq_te = to_seq(test_df["clean"].values)

lengths = list(map(len, seq_tr))
MAX_LEN = int(np.percentile(lengths, 95))
MAX_LEN = max(20, min(MAX_LEN, 200))
print("MAX_LEN:", MAX_LEN)

Xtr = pad_sequences(seq_tr, maxlen=MAX_LEN, padding='post', truncating='post')
Xva = pad_sequences(seq_va, maxlen=MAX_LEN, padding='post', truncating='post')
Xte = pad_sequences(seq_te, maxlen=MAX_LEN, padding='post', truncating='post')

ytr = train_df["y"].values.astype(int)
yva = val_df["y"].values.astype(int)
yte = test_df["y"].values.astype(int)

model = Sequential([
    Embedding(input_dim=VOCAB+1, output_dim=128, input_length=MAX_LEN),
    Conv1D(128, 5, activation='relu'),
    GlobalMaxPooling1D(),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='binary_crossentropy', metrics=['accuracy'])

es = EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
hist = model.fit(Xtr, ytr, validation_data=(Xva, yva),
                 epochs=10, batch_size=128, callbacks=[es], verbose=1)

# Évaluation DL
proba_dl = model.predict(Xte, verbose=0).ravel()
pred_dl  = (proba_dl >= 0.5).astype(int)

from sklearn.metrics import precision_recall_fscore_support
acc = (pred_dl == yte).mean()
pr, rc, f1, _ = precision_recall_fscore_support(yte, pred_dl, average='binary', zero_division=0)
from sklearn.metrics import roc_auc_score
dl_scores_test = {"accuracy":float(acc), "precision":float(pr), "recall":float(rc), "f1":float(f1), "roc_auc":float(roc_auc_score(yte, proba_dl))}
print("Scores test (DL):", dl_scores_test)

print("\nClassification report (DL):\n",
      classification_report(yte, pred_dl, target_names=["négatif","positif"], digits=4))

cm_dl = confusion_matrix(yte, pred_dl)
cm_dl


MAX_LEN: 175


Epoch 1/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 91s 285ms/step - accuracy: 0.6872 - loss: 0.5482 - val_accuracy: 0.9050 - val_loss: 0.2302
Epoch 2/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 85s 270ms/step - accuracy: 0.9132 - loss: 0.2195 - val_accuracy: 0.9072 - val_loss: 0.2367
Epoch 3/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 143s 273ms/step - accuracy: 0.9606 - loss: 0.1102 - val_accuracy: 0.9048 - val_loss: 0.2883
Epoch 4/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 143s 278ms/step - accuracy: 0.9817 - loss: 0.0577 - val_accuracy: 0.9024 - val_loss: 0.3506
Scores test (DL): {'accuracy': 0.9068, 'precision': 0.9081059390048154, 'recall': 0.9052, 'f1': 0.9066506410256411, 'roc_auc': 0.9683135800000001}

Classification report (DL):
               precision    recall  f1-score   support

     négatif     0.9055    0.9084    0.9069      5000
     positif     0.9081    0.9052    0.9067      5000

    accuracy                         0.9068     10000
   macro avg     0.9068    0.9068    0.9068     10000
weighted avg     0.9068  

array([[4542,  458],
       [ 474, 4526]])

In [8]:

# Comparatif
import pandas as pd
summary = pd.DataFrame([
    {"modèle":"ML", **ml_scores_test},
    {"modèle":"DL (CNN 1D)", **dl_scores_test}
])
summary


,modèle,accuracy,precision,recall,f1,roc_auc
0,ML (meilleur),0.9347,0.933573,0.9360,0.934785,0.983351
1,DL (CNN 1D),0.9068,0.908106,0.9052,0.906651,0.968314


# Rapport — Mise à jour finale

## Contexte
Ce travail porte sur des avis clients en français issus du dataset Allociné. L’objectif est de classifier automatiquement les critiques en deux classes : positif ou négatif, dans une optique de suivi de la satisfaction et de modération.

## Données
Le corpus contient des critiques de longueur variable, globalement équilibrées entre les deux classes. Les champs titre et review ont été fusionnés pour enrichir l’information textuelle par avis. Un échantillonnage équilibré a été utilisé pour maîtriser les temps de calcul (≈20k avis par classe pour l’entraînement).

## Prétraitement
Prétraitement simple et ciblé : passage en minuscules, suppression d’URL, emails, hashtags et mentions, retrait de la ponctuation et des chiffres, filtrage d’une courte liste de stopwords français. L’objectif est de conserver le vocabulaire porteur de polarité (adjectifs, expressions) sans appauvrir le texte.

## Représentation et modèles
Deux familles ont été comparées.

- Machine Learning classique  
  Représentation TF-IDF avec n-grams (1–2), vocabulaire plafonné (≈80k). Modèles : Logistic Regression et Linear SVM. Sélection du meilleur sur F1 validation, puis ré-entraînement sur train+val avant évaluation finale sur test.

- Deep Learning  
  Tokenisation  embedding (128D)  CNN 1D (Conv1D  GlobalMaxPooling  Dense). Early stopping pour limiter l’overfitting. Séquences tronquées à une longueur maximale déterminée à partir du 95e percentile (bornée entre 20 et 200 tokens).

## Évaluation — métriques et sélection de modèle
Métriques utilisées : accuracy, precision, recall, F1-score, ROC-AUC, et matrice de confusion.

### Validation (comparaison ML)
- Logistic Regression (val) : accuracy 0,9286 ; precision 0,9204 ; recall 0,9384 ; F1 0,9293 ; ROC-AUC 0,9783.
- Linear SVM (val) : accuracy 0,9326 ; precision 0,9256 ; recall 0,9408 ; F1 0,9331 ; ROC-AUC 0,9797.

La légère avance sur la F1-score et la stabilité de la SVM en haute dimension justifient la sélection de **Linear SVM** pour le test.

### Test (meilleur ML : Linear SVM)
- accuracy 0,9347 ; precision 0,9336 ; recall 0,9360 ; F1 0,9348 ; ROC-AUC 0,9834.  
- classification report (classes équilibrées) :
  - négatif : precision 0,9358 ; recall 0,9334 ; F1 0,9346 (support 5000)
  - positif : precision 0,9336 ; recall 0,9360 ; F1 0,9348 (support 5000)
- matrice de confusion (true rows × predicted cols) :
  - [[4667, 333],
     [ 320, 4680]]
  - 333 avis négatifs prédits positifs ; 320 avis positifs prédits négatifs.

Remarques :
- Les erreurs sont relativement symétriques entre classes, ce qui est cohérent avec un dataset équilibré et une décision proche d’un seuil neutre.
- Le ROC-AUC élevé indique un bon pouvoir de séparation des classes et laisse la possibilité d’ajuster le seuil si l’objectif opérationnel privilégie le rappel ou la précision.

### Test (DL : CNN 1D)
- accuracy 0,9068 ; precision 0,9081 ; recall 0,9052 ; F1 0,9067 ; ROC-AUC 0,9683.

Remarque :
- Le CNN 1D est compétitif sans réglages lourds mais reste en-dessous de la SVM TF-IDF sur ce corpus. Pour des avis courts/moyens, des représentations lexicales explicites (n-grams) fonctionnent très bien ; la CNN tire moins parti du contexte long et peut souffrir d’un vocabulaire très varié.

## Analyse détaillée des résultats
1. Effet de la représentation  
   TF-IDF 1–2-grams capture efficacement des motifs courts fréquents (adjectifs, tournures comme “très bon”, “sans intérêt”). Cela suffit à séparer nettement la polarité sur Allociné. L’approche embeddings + CNN 1D reste pertinente, mais nécessite souvent davantage de réglages (taille du vocabulaire, longueur de séquence, régularisation) et/ou des embeddings pré-entraînés pour rivaliser.

2. Biais et erreurs typiques  
   - Ironie/sarcasme et formulations ambiguës : quelques avis négatifs prédits positifs contiennent un vocabulaire élogieux tourné au second degré.  
   - Avis très courts ou neutres : manque d’indices de polarité, d’où des confusions.  
   - Références culturelles/noms propres : le modèle peut sur-interpréter certains termes propres à des franchises ou acteurs.

3. Stabilité et généralisabilité  
   Les scores proches entre validation et test suggèrent une bonne généralisation. La SVM profite de l’espace TF-IDF haute dimension et d’un coût d’entraînement raisonnable. Le ROC-AUC supérieur à 0,98 confirme la marge pour adapter le seuil selon un cahier des charges (par exemple favoriser le rappel pour capter plus d’avis réellement négatifs en modération).

4. Interprétabilité
   La SVM linéaire et la régression logistique restent interprétables via les poids de features. Une inspection rapide des poids positifs/négatifs les plus marqués montre des adjectifs et bigrams attendus pour chaque polarité, ce qui est utile pour un retour métier.

## Limites
- Couverture limitée du contexte long, surtout pour des critiques plus développées.
- Prétraitement volontairement sobre ; certaines variantes (lemmatisation, gestion plus fine de la négation) pourraient réduire des erreurs fines.
- Le modèle DL a été entraîné avec une architecture simple et des embeddings appris from scratch, sans pré-entraînement.

## Pistes d’amélioration
- Étendre les n-grams à 1–3 et tester une régularisation plus fine (C pour SVM) avec recherche d’hyperparamètres.  
- Ajuster le seuil de décision selon l’usage (par ex. maximiser F1 ou privilégier recall sur la classe négative en modération).  
- Utiliser des modèles pré-entraînés de type CamemBERT/BERT (fine-tuning léger), souvent supérieurs sur des nuances sémantiques et l’ironie.  
- Ajouter une détection de phrases peu informatives et une gestion explicite de la négation.

## Conclusion
Le pipeline met en évidence une performance solide des approches classiques sur ce jeu de données. **Linear SVM avec TF-IDF** fournit le meilleur compromis précision/robustesse (accuracy ≈ 0,935 ; F1 ≈ 0,935 ; ROC-AUC ≈ 0,983), avec des erreurs symétriques et interprétables. Le **CNN 1D** atteint des scores corrects mais inférieurs dans cette configuration. Pour aller plus loin, l’usage d’un modèle pré-entraîné francophone (CamemBERT) et un réglage de seuil orienté métier sont des prolongements naturels.